# Example 2 — Solid-body rotation basic state

Manuscript Sect. 4.2. A **solid-body super-rotation** zonal flow (u = u₀·cos φ) lets a localised off-equatorial heat source radiate a stationary **Rossby-wave train** that arcs poleward and eastward — the idealised teleconnection test.

## 1. Set up the model (T42 L20, Isca/GFDL core)

In [ ]:
# --- locate the LBM_isca package and the shared input data -------------------
import os, sys, types
import numpy as np

root = os.path.abspath("")
while root != os.path.dirname(root):
    if os.path.isdir(os.path.join(root, "lbm_isca")):
        break
    root = os.path.dirname(root)
sys.path.insert(0, root)
DATA = os.path.join(os.path.dirname(root), "LBM_python_GPU")  # bs.*.bin / frc.*.bin

from lbm_isca import Config, LBM, Spectral, VerticalGrid, asnumpy, gpu_available
from lbm_isca.forcing import SteadyForcing, MJOForcing
from lbm_isca.basicstate import load_basic_state

DEVICE = "gpu" if gpu_available() else "cpu"
sp = Spectral(device=DEVICE)
vg = VerticalGrid(device=DEVICE)
lon, lat, sig = asnumpy(sp.lon_deg), asnumpy(sp.lat_deg), asnumpy(vg.sig)
print(f"LBM_isca  T{sp.nmax}  {sp.nlon}x{sp.nlat}  L{vg.kmax}  backend={DEVICE}")

## 2. Basic state

The basic state is a **solid-body rotation**: zonal wind u = 25·cos(latitude) m s⁻¹, otherwise isothermal 250 K at 1000 hPa. The mean flow is what bends the forced response into a wave train.

> **This is the only block that changes between the three examples — swap it to run a different case.**

In [ ]:
# BASIC STATE — solid-body rotation (u = u0 * cos(lat))
shape = (sp.nlat, sp.nlon, vg.kmax)
coslat = np.sqrt(1.0 - asnumpy(sp.mu) ** 2)
u = np.broadcast_to(25.0 * coslat[:, None, None], shape).copy()
bs = types.SimpleNamespace(
    u=u, v=np.zeros(shape),
    t=np.full(shape, 250.0), q=np.zeros(shape),
    ps=np.full((sp.nlat, sp.nlon), 1000.0))

## 3. Forcing

Steady Gaussian heating centred at 15°N on the date line (2 K day$^{-1}$, half-widths 15° lon / 8° lat) — an off-equatorial source so the mean flow can refract the Rossby response.

In [ ]:
frc = SteadyForcing.zeros(sp.nlat, sp.nlon, vg.kmax)
dl = (lon[None, :] - 180.0 + 180.0) % 360.0 - 180.0
hxy = np.exp(-(dl / 15.0) ** 2 - ((lat[:, None] - 15.0) / 8.0) ** 2)
vz = np.sin(np.pi * (1.0 - sig)) * (sig > 0.1)
frc.t = hxy[:, :, None] * vz[None, None, :] * (2.0 / 86400.0)  # K/s

## 4. Integrate

In [ ]:
model = LBM(bs, sp, vg, Config(ndays=25.0))
model.run(steady_forcing=frc, verbose=True)

## 5. Look at the response

In [ ]:
# --- quick look: upper-level (sigma ~ 0.175, ~200 hPa) streamfunction --------
import matplotlib.pyplot as plt

k = int(np.argmin(np.abs(sig - 0.175)))
psi = asnumpy(model.ga.psi[:, :, k]) if hasattr(model.ga, "psi") \
      else asnumpy(model.diagnostics()["psi"][:, :, k])

fig, ax = plt.subplots(figsize=(10, 4.5))
vmax = np.abs(psi).max() or 1.0
cf = ax.contourf(lon, lat, psi, levels=np.linspace(-vmax, vmax, 21),
                 cmap="RdBu_r", extend="both")
ax.contour(lon, lat, psi, levels=np.linspace(-vmax, vmax, 21),
           colors="k", linewidths=0.3)
ax.set_ylim(-80, 80); ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(r"Upper-level ($\sigma\approx0.175$) streamfunction $\psi'$"
             r" — stationary Rossby-wave train", loc="left")
fig.colorbar(cf, ax=ax, pad=0.02, fraction=0.025)
fig.tight_layout(); plt.show()

## 6. (optional) Save gridded output

In [ ]:
# --- (optional) write GrADS + netCDF output ----------------------------------
from lbm_isca.output import GradsWriter  # NetCDFWriter also available
os.makedirs(os.path.join(root, "out"), exist_ok=True)
# Re-run with a writer attached if you want the full gridded history on disk:
#   w = GradsWriter(os.path.join(root, "out", "solidbody.grd"), sp, vg)
#   model.run(steady_forcing=frc, writer=w)
print("final-state grid fields available as model.ga.u / .v / .t / .psi ...")